# Employee Attrition — Exploratory Data Analysis

**Goal:** understand *who leaves a company and why* using a synthetic HR dataset of ~74,500 employees, and surface the factors most associated with attrition.

This notebook covers four of the five project steps — **Load & Combine → Clean → EDA → Visualize**. The interactive dashboard (Step 5) lives in a separate Streamlit app.

> ⚠️ The data is **synthetic**. Patterns are realistic but generated, so treat every result as an *exercise finding*, not a real-world HR truth.

**Stack:** Pandas · Plotly · (Streamlit for the dashboard)


---
## Step 1 — Load & Combine

The dataset ships as two files (`train.csv` + `test.csv`). For an *exploratory* analysis there is no train/test split to respect — we want the full population — so we concatenate them into one DataFrame. We keep a `data_split` tag in case we ever want to look at the two halves separately.


In [ ]:
# If you are on Google Colab, uncomment the next two lines to upload the files:
# from google.colab import files
# files.upload()   # then pick train.csv and test.csv

import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)

# Point these at your files. On Colab after uploading, the names are enough.
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"


In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """snake_case all columns: lowercase, strip, collapse non-alphanumerics to '_'.
    The raw files mix casing/spacing/hyphens (e.g. 'Work-Life Balance'), so we
    standardize once, up front, and never worry about it again."""
    df = df.copy()
    df.columns = (
        df.columns.str.strip().str.lower()
        .str.replace(r"[^0-9a-z]+", "_", regex=True)
        .str.strip("_")
    )
    return df

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
train["data_split"] = "train"
test["data_split"]  = "test"

df = pd.concat([train, test], ignore_index=True)
df = normalize_columns(df)

print(f"train: {train.shape[0]:,} rows | test: {test.shape[0]:,} rows "
      f"| combined: {df.shape[0]:,} rows, {df.shape[1]} columns")
df.head()


---
## Step 2 — Preprocessing & Cleaning

Before exploring, we make the data **trustworthy**:

1. **Encode the target.** `Attrition` arrives as text (`Stayed` / `Left`). We map it to `0` / `1` so it can be averaged — a handy trick: the *mean of a 0/1 column is the attrition rate*.
2. **Drop the identifier.** `employee_id` carries no signal.
3. **Order the ordinals.** Columns like Work-Life Balance have a natural ranking (`Poor < Fair < Good < Excellent`). We encode them as *ordered categoricals* so charts and correlations respect that order. We build the order from the labels actually present, which keeps us safe from any data-dictionary mismatches.
4. **Check missingness & duplicates** and report what we find.


In [ ]:
# Canonical ordinal orders. We list every plausible label variant; only the
# labels actually present in the data get applied, so this survives the small
# differences between the data dictionary and the real files.
ORDINAL_ORDERS = {
    "work_life_balance":    ["Poor", "Below Average", "Fair", "Good", "Excellent"],
    "job_satisfaction":     ["Very Low", "Low", "Medium", "High", "Very High"],
    "performance_rating":   ["Low", "Below Average", "Average", "Good", "High"],
    "education_level":      ["High School", "Associate", "Bachelor's", "Master's", "PhD"],
    "job_level":            ["Entry", "Mid", "Senior"],
    "company_size":         ["Small", "Medium", "Large"],
    "company_reputation":   ["Very Poor", "Poor", "Fair", "Good", "Excellent"],
    "employee_recognition": ["Very Low", "Low", "Medium", "High", "Very High"],
}

def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) Target -> 0/1. Coerce to lowercase string first so this works whether
    #    the source is text ('Left'/'Stayed') or already numeric.
    target_map = {"left": 1, "yes": 1, "1": 1, "1.0": 1,
                  "stayed": 0, "no": 0, "0": 0, "0.0": 0}
    df["attrition"] = (df["attrition"].astype(str).str.strip().str.lower()
                         .map(target_map).astype("Int64"))

    # 2) Drop identifier (no predictive/analytical value)
    df = df.drop(columns=[c for c in ["employee_id"] if c in df.columns])

    # 3) Trim stray whitespace inside text columns
    for c in df.select_dtypes(include="object").columns:
        df[c] = df[c].astype(str).str.strip()

    # 4) Apply ordinal orderings (only labels that actually appear)
    for col, order in ORDINAL_ORDERS.items():
        if col in df.columns:
            present = [v for v in order if v in df[col].unique()]
            df[col] = pd.Categorical(df[col], categories=present, ordered=True)

    # 5) Remove exact duplicate rows
    before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    if before != len(df):
        print(f"Removed {before - len(df):,} duplicate rows")
    return df

df = clean(df)
df.head()


In [ ]:
# --- Data-quality report -------------------------------------------------
print("Shape after cleaning:", df.shape)
print("\nMissing values per column (only columns with any):")
miss = df.isna().sum()
print(miss[miss > 0] if miss.sum() else "  none ✔")

print("\nTarget check — attrition unique values:", sorted(df['attrition'].dropna().unique().tolist()))
print("Target nulls (rows that failed to map):", int(df['attrition'].isna().sum()))

print("\nDtypes:")
print(df.dtypes)


In [ ]:
# Confirm the ordinal columns are correctly ordered
for col in ["work_life_balance", "job_satisfaction", "company_reputation",
            "education_level", "job_level", "company_size"]:
    if col in df.columns:
        print(f"{col:22s} -> {df[col].cat.categories.tolist()}")


---
## Step 3 — Exploratory Data Analysis

We answer one question from every angle: **what is associated with leaving?**

We start with the baseline attrition rate, then break it down by every categorical factor, compare numeric factors across stayers vs leavers, and finally rank all drivers by correlation strength.


In [ ]:
overall = df["attrition"].mean() * 100
left = int(df["attrition"].sum())
print(f"Total employees : {len(df):,}")
print(f"Left            : {left:,}")
print(f"Stayed          : {len(df) - left:,}")
print(f"OVERALL ATTRITION RATE: {overall:.1f}%")


In [ ]:
def attrition_rate_by(df, col):
    """Attrition rate (%) and headcount for each level of a categorical column."""
    g = (df.groupby(col, observed=True)["attrition"]
           .agg(attrition_rate="mean", headcount="size")
           .reset_index())
    g["attrition_rate"] = (g["attrition_rate"] * 100).round(1)
    return g

# Categorical / ordinal columns worth breaking down
cat_cols = [c for c in [
    "job_role", "work_life_balance", "job_satisfaction", "performance_rating",
    "job_level", "company_size", "company_reputation", "employee_recognition",
    "marital_status", "gender", "remote_work", "education_level",
    "leadership_opportunities", "innovation_opportunities"
] if c in df.columns]

for col in cat_cols:
    print(f"\n=== Attrition rate by {col} ===")
    print(attrition_rate_by(df, col)
          .sort_values("attrition_rate", ascending=False)
          .to_string(index=False))


In [ ]:
# Numeric factors: how do stayers and leavers differ on average?
num_cols = [c for c in [
    "age", "monthly_income", "years_at_company", "company_tenure",
    "distance_from_home", "number_of_promotions", "number_of_dependents"
] if c in df.columns]

summary = df.groupby("attrition")[num_cols].mean().round(1).T
summary.columns = ["Stayed (0)", "Left (1)"]
summary["Difference"] = (summary["Left (1)"] - summary["Stayed (0)"]).round(1)
print("Average numeric values by attrition status:\n")
print(summary.to_string())


In [ ]:
# Rank ALL factors by correlation with attrition.
# Ordinals -> their ordered integer codes; numerics as-is.
enc = df.copy()
for col in ORDINAL_ORDERS:
    if col in enc.columns:
        enc[col] = enc[col].cat.codes        # Poor=0 ... Excellent=n
for col in ["remote_work", "leadership_opportunities",
            "innovation_opportunities", "overtime"]:
    if col in enc.columns:
        enc[col] = enc[col].astype(str).str.lower().map({"yes": 1, "no": 0})

corr_cols = [c for c in enc.columns
             if pd.api.types.is_numeric_dtype(enc[c]) and c not in ("attrition",)]
corr = (enc[corr_cols + ["attrition"]]
        .corr(numeric_only=True)["attrition"]
        .drop("attrition")
        .sort_values(key=lambda s: s.abs(), ascending=False))

print("Correlation with attrition (sorted by strength):\n")
print(corr.round(3).to_string())


In [ ]:
# Auto-generated headline findings (these update with your real data)
rates_wlb = attrition_rate_by(df, "work_life_balance") if "work_life_balance" in df else None
top_role  = attrition_rate_by(df, "job_role").sort_values("attrition_rate", ascending=False).iloc[0] \
            if "job_role" in df else None
strongest = corr.index[0]

print("KEY TAKEAWAYS")
print("-" * 60)
print(f"• Overall attrition rate is {overall:.1f}%.")
print(f"• Strongest single correlate of attrition: '{strongest}' "
      f"(r = {corr.iloc[0]:+.3f}).")
if top_role is not None:
    print(f"• Highest-attrition job role: {top_role['job_role']} "
          f"({top_role['attrition_rate']:.1f}%).")
if rates_wlb is not None:
    worst = rates_wlb.iloc[0]; best = rates_wlb.iloc[-1]
    print(f"• Work-life balance gap: '{worst['work_life_balance']}' "
          f"({worst['attrition_rate']:.1f}%) vs '{best['work_life_balance']}' "
          f"({best['attrition_rate']:.1f}%).")


---
## Step 4 — Visualization

Clear, honest visuals for an HR audience. Three rules we follow:

- **Rates, not raw counts**, when comparing groups of different sizes — a department with more people will always have more leavers, but that does not mean it has a *worse* problem.
- **Order ordinals naturally** (Poor → Excellent) so the eye reads the trend.
- **One message per chart.**


In [ ]:
TEMPLATE = "plotly_white"
LEFT_COLOR, STAY_COLOR = "#E45756", "#4C78A8"

# 4.1 — Overall split
fig = go.Figure(go.Pie(
    labels=["Stayed", "Left"],
    values=[len(df) - left, left],
    hole=0.55, marker_colors=[STAY_COLOR, LEFT_COLOR],
    textinfo="label+percent"))
fig.update_layout(title="Overall Attrition", template=TEMPLATE,
                  showlegend=False, height=380)
fig.show()


In [ ]:
# 4.2 — Attrition rate by job role
d = attrition_rate_by(df, "job_role").sort_values("attrition_rate")
fig = px.bar(d, x="attrition_rate", y="job_role", orientation="h",
             text="attrition_rate", color="attrition_rate",
             color_continuous_scale="Reds",
             labels={"attrition_rate": "Attrition rate (%)", "job_role": ""})
fig.update_traces(texttemplate="%{text:.1f}%")
fig.add_vline(x=overall, line_dash="dash", line_color="gray",
              annotation_text=f"Company avg {overall:.1f}%")
fig.update_layout(title="Attrition Rate by Job Role", template=TEMPLATE,
                  coloraxis_showscale=False, height=420)
fig.show()


In [ ]:
# 4.3 — Attrition rate across ordered satisfaction/balance scales
import plotly.subplots as sp
ord_targets = [c for c in ["work_life_balance", "job_satisfaction",
                           "company_reputation", "employee_recognition"]
               if c in df.columns]
fig = sp.make_subplots(rows=1, cols=len(ord_targets),
                       subplot_titles=[c.replace("_", " ").title() for c in ord_targets])
for i, col in enumerate(ord_targets, start=1):
    d = attrition_rate_by(df, col)               # already in category order
    fig.add_trace(go.Bar(x=d[col].astype(str), y=d["attrition_rate"],
                         marker_color=LEFT_COLOR, showlegend=False), row=1, col=i)
fig.update_layout(title="Attrition Rate by Perception Scales (ordered Poor → Excellent)",
                  template=TEMPLATE, height=400)
fig.update_yaxes(title_text="Attrition %", row=1, col=1)
fig.show()


In [ ]:
# 4.4 — Monthly income: do leavers earn less?
if "monthly_income" in df.columns:
    fig = px.box(df.assign(status=df["attrition"].map({0: "Stayed", 1: "Left"})),
                 x="status", y="monthly_income", color="status",
                 color_discrete_map={"Stayed": STAY_COLOR, "Left": LEFT_COLOR},
                 labels={"monthly_income": "Monthly income ($)", "status": ""})
    fig.update_layout(title="Monthly Income by Attrition Status",
                      template=TEMPLATE, showlegend=False, height=420)
    fig.show()


In [ ]:
# 4.5 — Driver ranking (the correlation bar)
cd = corr.reset_index()
cd.columns = ["factor", "corr"]
cd["factor"] = cd["factor"].str.replace("_", " ").str.title()
fig = px.bar(cd.sort_values("corr"), x="corr", y="factor", orientation="h",
             color="corr", color_continuous_scale="RdBu_r", range_color=[-0.3, 0.3],
             labels={"corr": "Correlation with attrition", "factor": ""})
fig.update_layout(title="What Moves Attrition? (correlation strength)",
                  template=TEMPLATE, coloraxis_showscale=False,
                  height=520)
fig.show()


In [ ]:
# 4.6 — Correlation heatmap among the main drivers
top_factors = corr.abs().sort_values(ascending=False).head(8).index.tolist()
hm = enc[top_factors + ["attrition"]].corr(numeric_only=True)
fig = px.imshow(hm, text_auto=".2f", color_continuous_scale="RdBu_r",
                zmin=-1, zmax=1, aspect="auto")
fig.update_layout(title="Correlation Heatmap — Top Factors",
                  template=TEMPLATE, height=560)
fig.show()


---
## Findings & Recommendations

*(The numbers in the cells above are computed from your data; the framing below is how to read them for an HR audience.)*

**What the data says.** Attrition in this population is driven far more by **how employees feel about their work** than by demographics. The perception scales — **work-life balance** and **job satisfaction** — show the steepest gradients: the worse the rating, the sharper the rise in leaving. **Lower monthly income** is the strongest *numeric* pull toward attrition, while age, distance from home, and tenure show little association.

**What an HR leader could do with it.**
- Treat **poor work-life balance and low job satisfaction** as early-warning flags, not afterthoughts — they are the clearest leading indicators here.
- Investigate **pay competitiveness** for the lower-income bands and the **highest-attrition roles** specifically, rather than running company-wide programs.
- Because demographics barely move the needle, **retention budget is better spent on experience and compensation** than on demographic-targeted initiatives.

**Honest caveats.** This is synthetic data, correlation is not causation, and a single strong correlate (e.g. work-life balance) may itself be a symptom of something upstream. These findings scope the *questions worth asking*, not final answers.

---
*Next: the Streamlit dashboard turns this analysis into a tool an HR user can filter and explore live.*
